# 04 Resultaten inlezen en weergeven

In dit script worden de modelresultaten weergegeven en geplot.

In [1]:
import shutil
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import hkvsobekpy
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point

: 

In [ ]:
%load_ext autoreload
%autoreload 2

### Inlezen meetlocaties en meetdata

Afvoermetingen

In [ ]:
def inlezen_csv_met_metadata(file_path: Path):
    meta = {}
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        if not line.startswith("#"):
            continue
        text = line.strip("#").strip()
        if ":" in text:
            key, value = text.split(":", 1)
            meta[key.strip()] = value.strip().strip("; ")

    header_idx = next(i for i, line in enumerate(lines) if "Tijdstip (UTC);Waarde" in line)

    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=header_idx + 1,
        header=None,
        names=["Tijdstip (UTC)", "Waarde"],
        decimal=",",
        skipinitialspace=True
    )

    df["Tijdstip (UTC)"] = pd.to_datetime(df["Tijdstip (UTC)"], utc=True)
    df["time"] = df["Tijdstip (UTC)"].dt.tz_convert("Europe/Amsterdam").dt.tz_localize(None)
    df["Waarde"] = df["Waarde"].replace("---", np.nan).str.replace(",", ".").astype(float)
    df = df.set_index("time")[["Waarde"]]
    df.columns = [csv_file.stem]
    x = float(meta["Postitie X"].split(";")[0].strip("; (RD)"))
    y = float(meta["Postitie Y"].split(";")[0].strip("; (RD)"))

    return meta, Point(x,y), df

In [ ]:
# INPUT vanuit WRIJ voor RR unpaved methode
path_dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\IngekomenDataBewerkt\\")
path_oplevering_amigo = Path(path_dir_data, "20260508_AMIGOtifs_output")
path_overig = Path(path_oplevering_amigo, "04_Overig")
path_maaiveld = Path(path_overig, "MV25_FILL.ASC")
maaiveld = rioxarray.open_rasterio(path_maaiveld)

dir_input = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")
dir_input_basis_data = dir_input / "basisdata"

project_areas_path = dir_input_basis_data / "gebieden.gpkg"
path_watergang = dir_input_basis_data / "watergang.gpkg"
path_afwateringseenheden = dir_input_basis_data / "afwateringseenheden.gpkg"
path_hoog_middel_laag = dir_input_basis_data / "hoog_middel_laag.tif"

project_areas = gpd.read_file(project_areas_path, layer="gebieden")
watergang = gpd.read_file(path_watergang)
afwateringseenheden = gpd.read_file(path_afwateringseenheden)
hoog_middel_laag = rioxarray.open_rasterio(path_hoog_middel_laag)

In [ ]:
dir_meetdata = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\IngekomenData\\20260520_meetdata_RR")

# bro_peilbuizen
dir_bro_peilbuizen = Path(dir_meetdata, "bro_peilbuizen")
path_bro_peilbuizen = Path(dir_bro_peilbuizen, "alle_gebieden_metadata_min3jaar.gpkg")
bro_peilbuizen = gpd.read_file(path_bro_peilbuizen)

path_bro_peilbuizen_gebied_1 = Path(dir_bro_peilbuizen, "gebied_bergerslag", "gebied_bergerslag_alles.csv")
path_bro_peilbuizen_gebied_2 = Path(dir_bro_peilbuizen, "gebied_pelgrim", "gebied_pelgrim_alles.csv")
path_bro_peilbuizen_gebied_3 = Path(dir_bro_peilbuizen, "gebied_watermolen", "gebied_watermolen_alles.csv")

bro_peilbuizen_gebied_1 = pd.read_csv(path_bro_peilbuizen_gebied_1, parse_dates=["datetime"]).set_index("datetime")
bro_peilbuizen_gebied_2 = pd.read_csv(path_bro_peilbuizen_gebied_2, parse_dates=["datetime"]).set_index("datetime")
bro_peilbuizen_gebied_3 = pd.read_csv(path_bro_peilbuizen_gebied_3, parse_dates=["datetime"]).set_index("datetime")

bro_peilbuizen_metingen = pd.concat([bro_peilbuizen_gebied_1, bro_peilbuizen_gebied_2, bro_peilbuizen_gebied_3])
bro_peilbuizen_metingen = bro_peilbuizen_metingen[bro_peilbuizen_metingen["tube_number"]==1]

In [ ]:
bro_peilbuizen["hoog_middel_laag"] = hoog_middel_laag.sel(band=1).sel(
    x=xr.DataArray(bro_peilbuizen.geometry.x, dims="z"),
    y=xr.DataArray(bro_peilbuizen.geometry.y, dims="z"),
    method="nearest"
).to_dataframe(name="hoog_middel_laag")["hoog_middel_laag"]

bro_peilbuizen["maaiveld"] = maaiveld.sel(band=1).sel(
    x=xr.DataArray(bro_peilbuizen.geometry.x, dims="z"),
    y=xr.DataArray(bro_peilbuizen.geometry.y, dims="z"),
    method="nearest"
).to_dataframe(name="maaiveld")["maaiveld"]

In [ ]:
bro_peilbuizen.head(2)

In [ ]:
bro_peilbuizen_metingen.head(2)

In [ ]:
bro_peilbuizen_afw_eenheden = bro_peilbuizen[["gmw_bro_id", "hoog_middel_laag", "maaiveld", "geometry"]].sjoin(afwateringseenheden[["GFEIDENT", "geometry"]])

### Selecteer welke modelresultaten (gebieden, scenario’s en periode) worden geanalyseerd

In [ ]:
# path to the package containing the data
dir_model_basis = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\")

# gebied = 0 # Oude IJssel
# gebied = 1 # West
# gebied = 2 # Centraal
# gebied = 3 # Oost

gebieden = {
    1: "gebied_pelgrim", 
    2: "gebied_bergerslag", 
    3: "gebied_watermolen"
}
scenarios = ["REF", "SCEN"]

base_scenario = "REF"

runs = {
    "basis_test": {"name": "Eerste test", "color": "#0072B2"},          # blauw
    "basis_test_restart": {"name": "Eerste test + restart", "color": "#56B4E9"},  # lichtblauw
    "basis_aangepaste_kD": {"name": "kD (20d) L=4xl", "color": "#D55E00"},         # oranje
    "basis_aangepaste_kD_restart": {"name": "kD (20d) L=4xl + restart", "color": "#E69F00"},  # goud/oranje
    "basis_L2": {"name": "kD (20d) L=2xl", "color": "#009E73"},          # groen
    "basis_L2_restart": {"name": "kD (20d) L=2xl + restart", "color": "#CC79A7"},  # paars/roze
    "basis_L2_INF": {"name": "kD(20d) L=2xl INF", "color": "#F0E442"},   # geel
    "basis_L2_INF_restart": {"name": "kD(20d) L=2xl INF + restart", "color": "#D55E00"}, 
    "basis_L4_INF_W": {"name": "kD(20d) L=4xl INF Wh", "color": "#0072B2"}, 
    "basis_L4_INF_W_restart": {"name": "kD(20d) L=4xl INF Wh + restart", "color": "#666666"}, 
}

# LONG RUN
# start_date = "2010-4-1"
# end_date = "2018-12-31"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# LONG TEST
start_date = "2015-07-1"
end_date = "2016-09-30"
seizoenen = ["zomer", "winter", "winter", "zomer"]
date_range = pd.date_range(start_date, end_date, freq="3MS")

# SMALL TEST
# start_date = "2012-4-1"
# end_date = "2012-4-12"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

In [ ]:
simulations_total = pd.DataFrame()

for gebied in gebieden:
    for scenario in scenarios:

        simulaties = pd.DataFrame()
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = (seizoenen * 100)[:len(date_range)]
        simulaties["scenario"] = scenario
        simulaties["gebied"] = gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)
        simulations_total = pd.concat([simulations_total, simulaties])

In [ ]:
simulations_total

INLEZEN ALLE RUNS, GEBIEDEN, SCENARIOS

In [ ]:
total_rr_gwl = dict()

for run, run_dict in runs.items():
    print(run)
    for gebied in gebieden:
        for scenario in scenarios:
            simulaties = simulations_total[(simulations_total["gebied"]==gebied) & (simulations_total["scenario"]==scenario)]
            rr_gwl = pd.DataFrame()
            for index, simulatie in simulaties.iterrows():
                print(run  + " - " + str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

                dir_model = Path(dir_model_basis, run, f"gebied_{simulatie.gebied}", simulatie.scenario)
                unpaved_rr_file = "upflowdt.his"

                path_unpaved_rr_file = Path(dir_model, simulatie.model_name, "rr", unpaved_rr_file)
                if not path_unpaved_rr_file.exists():
                    print(f"File {path_unpaved_rr_file} does not exist. Skipping.")
                    continue
                rr_his = hkvsobekpy.read_his.ReadMetadata(path_unpaved_rr_file)
                rr_results = rr_his.DataFrame()['Groundw.Level   [m] ']
                rr_gwl = pd.concat([rr_gwl, rr_results])
                total_rr_gwl[f"{run}_{gebied}_{scenario}"] = rr_gwl

### Plot het resultaat

TODO
- gemeten afvoeren gebied 3
- initiële waterstanden
- L=2x kleine l ipv L = 4x kleine l
- grafieken grondwaterstanden van alle meetpunten plus kaartje
- dynamische grafieken

- grafieken referentie vs scenario: aangepaste kD met initiële grondwaterstanden
- harm maakt overzicht van uren, inspanningen en overdracht.

wibo gaat scenario aanpassen en dan de hele trein weer doen.

In [ ]:
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")

df_unpaved_zomers = pd.DataFrame()
df_unpaved_winters = pd.DataFrame()
df_ernst_zomers = pd.DataFrame()
df_ernst_winters = pd.DataFrame()

for gebied in [1,2,3]:
    dir_gebied = Path(dir_data, "rr_input_area_scenario", f"gebied_{gebied}")
    dir_scenarios = Path(dir_data, "rr_input_scenarios")
    dir_model_gebied = Path(dir_model_basis, f"gebied_{gebied}")

    gebied = gpd.read_file(dir_gebied / f"gebied.gpkg", layer=f"gebied")
    watergang = gpd.read_file(dir_gebied / f"watergang.gpkg", layer=f"watergang")
    afwateringseenheden = gpd.read_file(dir_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden")

    #Read modelinput
    df_unpaved_zomer = pd.read_csv(dir_gebied / scenario / f"df_unpaved_zomer.csv")
    df_unpaved_winter = pd.read_csv(dir_gebied / scenario / f"df_unpaved_winter.csv")
    df_ernst_zomer = pd.read_csv(dir_gebied / scenario / f"df_ernst_zomer.csv")
    df_ernst_winter = pd.read_csv(dir_gebied / scenario / f"df_ernst_winter.csv")

    df_unpaved_winters = pd.concat([df_unpaved_winters, df_unpaved_winter])
    df_unpaved_zomers = pd.concat([df_unpaved_zomers, df_unpaved_zomer])
    df_ernst_winters = pd.concat([df_ernst_winters, df_ernst_winter])
    df_ernst_zomers = pd.concat([df_ernst_zomers, df_ernst_zomer])

In [ ]:
from pathlib import Path
from uuid import uuid4
import plotly.io as pio

def save_plotly_tabs_html(figures, titles, output_path="plotly_tabs.html"):
    """
    Save multiple Plotly figures into one self-contained HTML file with tabs.

    Parameters
    ----------
    figures : list
        List of Plotly Figure objects.
    titles : list of str
        Tab titles, same length as figures.
    output_path : str or Path
        Output HTML file path.
    """
    if len(figures) != len(titles):
        raise ValueError("figures and titles must have the same length")

    output_path = Path(output_path)

    # Unique IDs for tabs and panels
    tab_ids = [f"tab-{uuid4().hex}" for _ in figures]
    panel_ids = [f"panel-{uuid4().hex}" for _ in figures]

    # Plotly JS included only once
    plotly_js = pio.to_html(
        figures[0],
        full_html=False,
        include_plotlyjs="cdn"
    ).split('<div id="')[0]

    # Generate figure divs without Plotly JS
    figure_divs = []
    for fig, panel_id in zip(figures, panel_ids):
        fig_html = pio.to_html(
            fig,
            full_html=False,
            include_plotlyjs=False,
            div_id=panel_id
        )
        figure_divs.append(fig_html)

    # CSS + JS for tabs
    css = """
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 0;
        padding: 16px;
      }
      .tab-bar {
        display: flex;
        gap: 8px;
        border-bottom: 1px solid #ddd;
        margin-bottom: 16px;
        flex-wrap: wrap;
      }
      .tab-button {
        appearance: none;
        border: 1px solid #ccc;
        border-bottom: none;
        background: #f7f7f7;
        padding: 10px 14px;
        cursor: pointer;
        border-radius: 8px 8px 0 0;
        font: inherit;
      }
      .tab-button.active {
        background: white;
        border-color: #999;
        font-weight: 600;
      }
      .tab-panel {
        display: none;
      }
      .tab-panel.active {
        display: block;
      }
    </style>
    """

    js = f"""
    <script>
      function showTab(idx) {{
        const buttons = document.querySelectorAll('.tab-button');
        const panels = document.querySelectorAll('.tab-panel');

        buttons.forEach(btn => btn.classList.remove('active'));
        panels.forEach(p => p.classList.remove('active'));

        buttons[idx].classList.add('active');
        panels[idx].classList.add('active');

        // Resize the Plotly chart after becoming visible
        const plotDiv = panels[idx].querySelector('.plotly-graph-div');
        if (plotDiv && window.Plotly) {{
          window.Plotly.Plots.resize(plotDiv);
        }}
      }}

      document.addEventListener('DOMContentLoaded', () => {{
        showTab(0);
      }});
    </script>
    """

    # Build tab buttons and panels
    tab_buttons = "\n".join(
        f'<button class="tab-button" onclick="showTab({i})">{title}</button>'
        for i, title in enumerate(titles)
    )

    tab_panels = "\n".join(
        f'<div class="tab-panel" id="{panel_ids[i]}-wrap">{figure_divs[i]}</div>'
        for i in range(len(figures))
    )

    html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  {css}
  {plotly_js}
</head>
<body>
  <div class="tab-bar">
    {tab_buttons}
  </div>
  {tab_panels}
  {js}
</body>
</html>
"""

    output_path.write_text(html, encoding="utf-8")
    return output_path

In [ ]:
df_unpaved_winters["GFEIDENT"] = df_unpaved_winters.apply(lambda x: x["code"].split("_")[0], axis=1)
df_unpaved_zomers["GFEIDENT"] = df_unpaved_zomers.apply(lambda x: x["code"].split("_")[0], axis=1)

df_ernst_winters["GFEIDENT"] = df_ernst_winters.apply(lambda x: x["code"].split("_")[0], axis=1)
df_ernst_zomers["GFEIDENT"] = df_ernst_zomers.apply(lambda x: x["code"].split("_")[0], axis=1)

In [ ]:
df_unpaved_winters = df_unpaved_winters[["GFEIDENT", "surface_level", "boundary_waterlevel", "code"]]
df_unpaved_zomers = df_unpaved_zomers[["GFEIDENT", "surface_level", "boundary_waterlevel", "code"]]

df_ernst_zomers = df_ernst_zomers[["GFEIDENT", "lv", "code"]]
df_ernst_winters = df_ernst_winters[["GFEIDENT", "lv", "code"]]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# path to the package containing the data
dir_model_basis = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\")

start_plot = "2015-10-01"
end_plot = "2016-09-30"

selectie_gebieden = {
    1: {"ymax": 10}, 
    2: {"ymax": 2},
    3: {"ymax": 10},
}

selectie_runs = {
    "basis_test": {"name": "Eerste test", "color": "#0072B2"},  # blauw
    "basis_test_restart": {"name": "Eerste test + restart", "color": "#56B4E9"},  
    "basis_aangepaste_kD": {"name": "Nieuwe kD (20d) L=4xl", "color": "#D55E00"},  
    "basis_aangepaste_kD_restart": {"name": "Nieuwe kD (20d) L=4xl + restart", "color": "#E69F00"}, 
    "basis_L2": {"name": "Nieuwe kD (20d) L=2xl", "color": "#009E73"},  
    "basis_L2_restart": {"name": "Nieuwe kD (20d) L=2xl + restart", "color": "#CC79A7"}, 
}

selectie_scenario = {
    "REF": {"linestyle": "solid", "visible": True}, 
    "SCEN": {"linestyle": "dash", "visible": False},
}
include_metingen = True

metingen_instroom = True
metingen = True
figures = []
figures_titles = []

for i, bro_peilbuis in bro_peilbuizen_afw_eenheden.iterrows():
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=("RR-knoop - Hoog", "RR-knoop - Middel", "RR-knoop - Laag")
    )

    gmw_bro_id = bro_peilbuis["gmw_bro_id"]
    afw_eenheid = bro_peilbuis["GFEIDENT"]
    print(gmw_bro_id, afw_eenheid)

    def add_levels_to_figure(start_date, end_date, code, df_unpaved_winters, df_unpaved_zomers, df_ernst_winters, df_ernst_zomers):
        if code not in df_unpaved_winters["code"]:
            return

        df_unpaved_winter = df_unpaved_winters[df_unpaved_winters["code"]==code].iloc[0]
        df_unpaved_zomer = df_unpaved_zomers[df_unpaved_zomers["code"]==f"{afw_eenheid}_{level}"].iloc[0]
        df_ernst_winter = df_ernst_winters[df_ernst_winters["code"]==f"{afw_eenheid}_{level}"].iloc[0]
        df_ernst_zomer = df_ernst_zomer[df_ernst_zomer["code"]==f"{afw_eenheid}_{level}"].iloc[0]

        def add_seasonal_levels(years, winterlevel, summerlevel, color="blue", dash="dash"):
            for year in years:
                # Winter: 1/10 until 1/4
                fig.add_shape(
                    type="line",
                    x0=f"{year}-10-01",
                    x1=f"{year+1}-04-01",
                    y0=winterlevel,
                    y1=winterlevel,
                    line=dict(color=color, dash=dash),
                    row=i+1, col=1
                )

                # Summer: 1/4 until 1/10
                fig.add_shape(
                    type="line",
                    x0=f"{year}-04-01",
                    x1=f"{year}-10-01",
                    y0=summerlevel,
                    y1=summerlevel,
                    line=dict(color=color, dash=dash),
                    row=i+1, col=1
                )
        years = np.arange(int(start_date[:3])-1, int(end_date[:3])+1)
        add_seasonal_levels(years=years, winterlevel=df_unpaved_winter["boundary_waterlevel"], summerlevel=df_unpaved_zomer["boundary_waterlevel"], color="blue", dash="dash")

    bro_peilbuis_metingen = bro_peilbuizen_metingen[bro_peilbuizen_metingen["gmw_bro_id"]==gmw_bro_id][start_date:end_date]
    for i, level in enumerate(["hoog", "middel", "laag"]):
        value = 3-i
        add_levels_to_figure(
            start_date=start_date, 
            end_date=end_date,
            code=f"{afw_eenheid}_{level}",
            df_unpaved_winters=df_unpaved_winters,
            df_unpaved_zomers=df_unpaved_zomers,
            df_ernst_winters=df_ernst_zomers,
            df_ernst_zomers=df_ernst_zomers
        )
        if bro_peilbuis["hoog_middel_laag"] == value:
            fig.add_trace(
                go.Scatter(
                    x=bro_peilbuis_metingen.index, 
                    y=bro_peilbuis_metingen.value, 
                    mode="markers", 
                    name=f"Gemeten grondwaterstand peilbuis ({level})",
                    marker=dict(color="purple", size=2)
                ),
                row=i+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=[start_date, end_date],
                    y=[bro_peilbuis["maaiveld"], bro_peilbuis["maaiveld"]],
                    mode="lines",
                    name="Maaiveld meetpunt",
                    line=dict(color="purple", dash="dash")
                ),
                row=i+1, col=1
            )

    for scenario, scenario_dict in selectie_scenario.items():
        for run, run_dict in selectie_runs.items():
            for gebied in selectie_gebieden.keys():
                gw = total_rr_gwl[run + "_" + str(gebied) + "_" + scenario]
                cols_bro_peilbuis = [col for col in gw.columns if afw_eenheid in col]
                if cols_bro_peilbuis:
                    break
            legend_visible = True
            for i, level in enumerate(["hoog", "middel", "laag"]):
                col_level = f"unp_{afw_eenheid}_{level}"
                if col_level in gw:
                    gmw_bro_peilbuis = gw[col_level]
                    fig.add_trace(
                        go.Scatter(
                            x=gmw_bro_peilbuis.index, 
                            y=gmw_bro_peilbuis, 
                            mode="lines", 
                            legendgroup=f"{run}_{scenario}",
                            name=f"{scenario} = {run_dict['name']}",
                            showlegend=True if legend_visible else False,
                            visible=True if (scenario_dict["visible"] and "_restart" in run) else "legendonly",
                            line=dict(dash=scenario_dict["linestyle"], color=run_dict["color"])
                        ),
                        row=i+1, col=1
                    )
                    legend_visible = False

    fig.update_layout(
        template="simple_white",
        title={
            "text": f"<b>RR-modellering Oude IJssel - Grondwaterstand {gmw_bro_id} - afwateringseenheid {afw_eenheid}</b>",
            "x": 0.05,
            "xanchor": "left"
        },
        margin=dict(l=20, r=20, t=50, b=20),
        height=800,
    )

    fig.update_xaxes(showgrid=True, gridcolor="lightgray", range=[start_plot, end_plot])
    fig.update_yaxes(showgrid=True, gridcolor="lightgray")

    # fig.write_html(Path(dir_model_basis, f"grondwaterstand_{gmw_bro_id}.html"), include_plotlyjs="cdn")
    figures.append(fig)
    figures_titles.append(gmw_bro_id[-6:])

    fig.show()
    break

save_plotly_tabs_html(figures, figures_titles, output_path=Path(dir_model_basis, f"grondwaterstand_pilotgebieden_oude_ijssel.html"));

In [ ]:
bro_peilbuizen_afw_eenheden

In [ ]:
bro_peilbuis_metingen

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

start_date = "2015-10-01"
end_date = "2016-09-30"

selectie_gebieden = {
    1: {"ymax": 10}, 
    2: {"ymax": 2},
    3: {"ymax": 10},
}

selectie_runs = {
    "basis_test": {"name": "Eerste test", "color": "#0072B2"},  # blauw
    "basis_test_restart": {"name": "Eerste test + restart", "color": "#56B4E9"},  
    "basis_aangepaste_kD": {"name": "Nieuwe kD (20d) L=4xl", "color": "#D55E00"},  
    "basis_aangepaste_kD_restart": {"name": "Nieuwe kD (20d) L=4xl + restart", "color": "#E69F00"}, 
    "basis_L2": {"name": "Nieuwe kD (20d) L=2xl", "color": "#009E73"},  
    "basis_L2_restart": {"name": "Nieuwe kD (20d) L=2xl + restart", "color": "#CC79A7"}, 
}

selectie_scenario = {
    "REF": {"linestyle": "solid", "visible": True}, 
    "SCEN": {"linestyle": "dash", "visible": False},
}
include_metingen = True

fig = make_subplots(
    rows=len(selectie_gebieden), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03
)

metingen_instroom = True
metingen = True



dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")

for gebied in [1,2,3]:
    dir_gebied = Path(dir_data, "rr_input_area_scenario", f"gebied_{gebied}")
    dir_scenarios = Path(dir_data, "rr_input_scenarios")
    dir_model_gebied = Path(dir_model_basis, f"gebied_{gebied}")

    gebied = gpd.read_file(dir_gebied / f"gebied.gpkg", layer=f"gebied")
    watergang = gpd.read_file(dir_gebied / f"watergang.gpkg", layer=f"watergang")
    afwateringseenheden = gpd.read_file(dir_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden")

    #Read modelinput
    df_unpaved_zomer = pd.read_csv(dir_gebied / scenario / f"df_unpaved_zomer.csv")
    df_unpaved_winter = pd.read_csv(dir_gebied / scenario / f"df_unpaved_winter.csv")
    df_ernst_zomer = pd.read_csv(dir_gebied / scenario / f"df_ernst_zomer.csv")
    df_ernst_winter = pd.read_csv(dir_gebied / scenario / f"df_ernst_winter.csv")
    

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

start_date = "2015-10-01"
end_date = "2016-09-30"

selectie_gebieden = {
    1: {"ymax": 10}, 
    2: {"ymax": 2},
    3: {"ymax": 10},
}

selectie_runs = {
    "basis_test": {"name": "Eerste test", "color": "#0072B2"},  # blauw
    "basis_test_restart": {"name": "Eerste test + restart", "color": "#56B4E9"},  
    "basis_aangepaste_kD": {"name": "Nieuwe kD (20d) L=4xl", "color": "#D55E00"},  
    "basis_aangepaste_kD_restart": {"name": "Nieuwe kD (20d) L=4xl + restart", "color": "#E69F00"}, 
    "basis_L2": {"name": "Nieuwe kD (20d) L=2xl", "color": "#009E73"},  
    "basis_L2_restart": {"name": "Nieuwe kD (20d) L=2xl + restart", "color": "#CC79A7"}, 
}

selectie_scenario = {
    "REF": {"linestyle": "solid", "visible": True}, 
    "SCEN": {"linestyle": "dash", "visible": False},
}
include_metingen = True

fig = make_subplots(
    rows=len(selectie_gebieden), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03
)

metingen_instroom = True
metingen = True

for i_gebied, gebied in enumerate(selectie_gebieden.keys()):

    if include_metingen:
        instroompunten = afvoermeetlocaties_gebieden[gebied]["in"]
        instroom = afvoermetingen[start_date:end_date][instroompunten].sum(axis=1)
        uitstroompunten = afvoermeetlocaties_gebieden[gebied]["uit"]
        uitstroom = afvoermetingen[start_date:end_date][uitstroompunten].sum(axis=1)
        if instroompunten:
            fig.add_trace(
                go.Scatter(
                    x=instroom.index, 
                    y=instroom, 
                    mode="markers", 
                    name="Instroom gebied 3",
                    showlegend=metingen_instroom,
                    legendgroup="instroom+uitstroom",
                    visible="legendonly",
                    marker=dict(color="purple")
                ),
                row=i_gebied+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=uitstroom.index, 
                    y=uitstroom, 
                    mode="markers", 
                    name="Uitstroom gebied 3",
                    showlegend=metingen_instroom,
                    legendgroup="instroom+uitstroom",
                    visible="legendonly",
                    marker=dict(color="darkgreen")
                ),
                row=i_gebied+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=uitstroom.index, 
                    y=uitstroom-instroom.shift(3),
                    mode="markers", 
                    name="Uitstroom - instroom gebied",
                    legendgroup="metingen",
                    marker=dict(color="black")
                ),
                row=i_gebied+1, col=1
            )
            metingen_instroom = False

        else:
            fig.add_trace(
                go.Scatter(
                    x=uitstroom.index, 
                    y=uitstroom,
                    mode="markers", 
                    name="Uitstroom compleet gebied",
                    showlegend=metingen,
                    legendgroup="metingen",
                    marker=dict(color="black")
                ),
                row=i_gebied+1, col=1
            )
        metingen = False

for i_gebied, gebied in enumerate(selectie_gebieden.keys()):

    total_link_flows_figure = total_link_flows[start_date:end_date]
    for scenario, scenario_dict in selectie_scenario.items():
        for run, run_dict in selectie_runs.items():
            data = total_link_flows_figure[f"{run}_{gebied}_{scenario}"].copy()
            fig.add_trace(
                go.Scatter(
                    x=data.index, 
                    y=data, 
                    mode="lines", 
                    name=f"{scenario} - {run_dict['name']}",
                    legendgroup=f"{run}_{scenario}",
                    showlegend=True if i_gebied==0 else False,
                    visible=True if scenario_dict["visible"] else "legendonly",
                    line=dict(dash=scenario_dict["linestyle"], color=run_dict["color"])
                ),
                row=i_gebied+1, col=1
            )

fig.update_layout(
    template="simple_white",
    title="RR-modellering Oude IJssel - Afvoer voor pilotgebieden 1/2/3",
    margin=dict(l=20, r=20, t=40, b=20),
    height=350*len(selectie_gebieden),
    # legend=dict(
    #     groupclick="togglegroup",
    #     orientation="h",
    #     yanchor="top",
    #     y=-0.05,
    #     xanchor="center",
    #     x=0.5,
    #     entrywidth=250,
    #     entrywidthmode="pixels"
    # ),
)

fig.update_xaxes(showgrid=True, gridcolor="lightgray", range=[total_link_flows_figure.index[0], total_link_flows_figure.index[-1]])
for i_gebied, (gebied, gebied_dict) in enumerate(selectie_gebieden.items()):
    fig.update_yaxes(
        row=i_gebied+1, 
        col=1, 
        range=[0, gebied_dict["ymax"]],
        showgrid=True,
        gridcolor="lightgray",
        title_text=f"Afvoer gebied {gebied} [m3/s]"
    )

fig.write_html(Path(dir_model_basis, f"oude_ijssel_pilot_gebieden_afvoeren.html"), include_plotlyjs="cdn")
fig.show()